# Tratamento — Ranking mundial 50m livre masculino (piscina longa), 2010–2025

Entrada: PDFs do ranking da World Aquatics, baixados manualmente.

In [11]:
import re, unicodedata
from pathlib import Path

import pandas as pd
import pdfplumber

ANOS = range(2010, 2026)   # 2010–2025 (2009 excluído: supertrajes; 2026 ainda incompleto)
N = 200 
RAW = Path("data/raw")

# Coleta: abrir no navegador, trocando AAAA pelo ano:
# https://api.worldaquatics.com/fina/rankings/swimming/report/pdf?gender=M&distance=50&stroke=FREESTYLE&poolConfiguration=LCM&timesMode=BEST_TIMES&pageSize=200&year=AAAA
pdfs = {ano: RAW / f"50free_M_{ano}.pdf" for ano in ANOS}
faltando = [ano for ano, p in pdfs.items() if not p.exists() or p.stat().st_size == 0]
assert not faltando, f"PDFs ausentes ou vazios em {RAW}/: {faltando}"
print(f"{len(pdfs)} PDFs encontrados em {RAW}/")

16 PDFs encontrados em data/raw/


## 1. Extração do texto e parsing

In [12]:
INICIO = re.compile(r"^\d+\s+\d{0,2}:?\d{1,2}\.\d{2}\s+\d+\s")
LINHA = re.compile(
    r"^(?P<posicao>\d+)\s+(?P<tempo>\d{0,2}:?\d{1,2}\.\d{2})\s+(?P<pontos>\d+)\s+"
    r"(?P<nome>.+?)\s+(?:(?P<ano_nascimento>\d{4})\s+)?(?P<pais>[A-Z]{3})\s+(?:Yes|No)\s+"
    r"(?P<resto>.*?)\s*(?P<data>\d{2}/\d{2}/\d{4})\s*(?P<rodada>SF|H)?\s*(?P<split>S)?\s*$"
)

def tempo_em_s(t):
    m, _, s = t.rpartition(":")
    return (int(m) * 60 if m else 0) + float(s)

def parse_pdf(caminho):
    with pdfplumber.open(caminho) as pdf:
        linhas = [l.strip() for p in pdf.pages for l in (p.extract_text() or "").splitlines()]
    registros, buf = [], None
    for l in linhas:
        if INICIO.match(l):
            if buf: registros.append(buf)
            buf = l
        elif buf and l:
            buf += " " + l
    if buf: registros.append(buf)

    ok, falhas = [], []
    for r in registros:
        m = LINHA.match(r)
        (ok if m else falhas).append(m.groupdict() if m else r)
    return ok, falhas

def parse_teste():
    # linha no formato confirmado do relatório (exemplo fictício para 50 livre)
    exemplo = "12 28.96 905 LEANDRO Beloti Kornelius 2003 BRA Yes 2018 Brazilian Championships Rio de Janeiro BRA 23/04/2018 SF"
    d = LINHA.match(exemplo).groupdict()
    assert d["nome"] == "LEANDRO Beloti Kornelius" and d["ano_nascimento"] == "2003" and d["rodada"] == "SF"
    assert tempo_em_s("28.96") == 28.96 and tempo_em_s("1:02.50") == 62.5
parse_teste()

In [13]:
linhas, todas_falhas = [], {}
for ano, caminho in pdfs.items():
    ok, falhas = parse_pdf(caminho)
    for d in ok:
        d["ano"] = ano
    linhas += ok
    if falhas:
        todas_falhas[ano] = falhas

print(f"{len(linhas)} linhas lidas")
for ano, f in todas_falhas.items():
    print(f"\n{ano}: {len(f)} linha(s) não reconhecida(s):")
    for l in f: print("   ", l)

3199 linhas lidas

2020: 1 linha(s) não reconhecida(s):
    198 22.93 758 GUILLERMO RUBEN ZUNIGA Guillermo2 0R0u0ben ZMuEnXiga No Brazil Trophy 2020 Rio de Janeiro BRA 10/12/2020


Linhas com texto sobreposto no PDF que a regex não consegue ler são corrigidas à mão abaixo, a partir da leitura visual do PDF.

In [14]:
CORRECOES_MANUAIS = [
    # 2020, posição 198: nome, nascimento e país sobrepostos no PDF (conferir no PDF original)
    dict(ano=2020, posicao="198", tempo="22.93", pontos="758", nome="ZUNIGA Guillermo Ruben",
         ano_nascimento="2000", pais="MEX", resto="Brazil Trophy 2020 Rio de Janeiro BRA",
         data="10/12/2020", rodada=None, split=None),
]
linhas += CORRECOES_MANUAIS
print(f"{len(linhas)} linhas após correções manuais")

3200 linhas após correções manuais


## 2. Tabela final

In [15]:
def normaliza(nome):
    s = unicodedata.normalize("NFKD", nome).encode("ascii", "ignore").decode()
    return re.sub(r"[^a-z ]", "", s.lower()).strip()

df = pd.DataFrame(linhas)
df["posicao"] = df["posicao"].astype(int)
df = df[df["posicao"] <= N]                      # empates podem passar de N
df["tempo_s"] = df["tempo"].map(tempo_em_s)
df["ano_nascimento"] = pd.to_numeric(df["ano_nascimento"], errors="coerce").astype("Int64")
df["idade"] = df["ano"] - df["ano_nascimento"]
df["data_prova"] = pd.to_datetime(df["data"], format="%d/%m/%Y")
df["competicao"] = df["resto"].str.strip()
df["rodada"] = df["rodada"].fillna("")
df["split"] = df["split"].notna()
df["atleta_id"] = df["nome"].map(normaliza) + "_" + df["ano_nascimento"].astype(str)

df = df[["ano", "posicao", "atleta_id", "nome", "ano_nascimento", "idade",
         "tempo_s", "data_prova", "competicao", "rodada", "split"]].sort_values(["ano", "posicao"])
df.head()

,ano,posicao,atleta_id,nome,ano_nascimento,idade,tempo_s,data_prova,competicao,rodada,split
0,2010,1,bousquet frederick_1981,BOUSQUET Frederick,1981,29,21.36,2010-08-14,2010 European Aquatics Championships (50m) Bud...,SF,False
1,2010,2,adrian nathan_1988,ADRIAN Nathan,1988,22,21.55,2010-08-21,11th Pan Pacific Championships 2010 Irvine USA,,False
2,2010,3,cielo filho cesar_1987,CIELO FILHO Cesar,1987,23,21.57,2010-08-21,11th Pan Pacific Championships 2010 Irvine USA,,False
3,2010,4,nystrand stefan_1981,NYSTRAND Stefan,1981,29,21.69,2010-08-15,2010 European Aquatics Championships (50m) Bud...,,False
4,2010,5,gilot fabien_1984,GILOT Fabien,1984,26,21.75,2010-08-14,2010 European Aquatics Championships (50m) Bud...,SF,False


## 3. Verificações

In [16]:
# (a) cada ano deve ter ~N linhas
print(df.groupby("ano").size().to_string())

# (b) a data da prova deve cair dentro do próprio ano (confirma que o filtro `year` funcionou)
fora = df[df["data_prova"].dt.year != df["ano"]]
print(f"\nProvas fora do ano do ranking: {len(fora)}")

# (c) sem ano de nascimento -> sem idade
print(f"Linhas sem ano de nascimento: {df['ano_nascimento'].isna().sum()}")

# (d) mesmo atleta aparecendo duas vezes no mesmo ano (não deveria, em BEST_TIMES)
dup = df[df.duplicated(["ano", "atleta_id"], keep=False)]
print(f"Duplicatas atleta/ano: {len(dup)}")

# (e) possível mesmo atleta com grafias diferentes: mesmo sobrenome + mesmo ano de nascimento
df["_sobrenome"] = df["nome"].str.split().str[0].map(normaliza)
suspeitos = (df.groupby(["_sobrenome", "ano_nascimento"])["atleta_id"].nunique()
               .loc[lambda s: s > 1])
print(f"Grupos sobrenome+nascimento com mais de um id (revisar à mão): {len(suspeitos)}")
display(df[df.set_index(["_sobrenome", "ano_nascimento"]).index.isin(suspeitos.index)]
          [["ano", "nome", "ano_nascimento", "atleta_id"]].sort_values("nome"))
df = df.drop(columns="_sobrenome")

ano
2010    200
2011    200
2012    200
2013    200
2014    200
2015    200
2016    200
2017    200
2018    200
2019    200
2020    200
2021    200
2022    200
2023    200
2024    200
2025    200

Provas fora do ano do ranking: 0
Linhas sem ano de nascimento: 11
Duplicatas atleta/ano: 4
Grupos sobrenome+nascimento com mais de um id (revisar à mão): 18


,ano,nome,ano_nascimento,atleta_id
2431,2022,ARMSTRONG Hunter,2001,armstrong hunter_2001
3021,2025,ARMSTRONG Hunter,2001,armstrong hunter_2001
1947,2019,ARMSTRONG Hunter,2001,armstrong hunter_2001
2664,2023,ARMSTRONG Hunter,2001,armstrong hunter_2001
2948,2024,ARMSTRONG Jack,2001,armstrong jack_2001
...,...,...,...,...
3140,2025,YANG William,1998,yang william_1998
3037,2025,YOUNG David,2005,young david_2005
3087,2025,YOUNG Tolu,2005,young tolu_2005
2939,2024,YURKEVYCH Yurii,1998,yurkevych yurii_1998


Se a verificação (e) mostrar a mesma pessoa com duas grafias, unifique os ids no dicionário abaixo e rode a célula.

In [ ]:
UNIFICAR = {
    # "id_variante": "id_canonico",
}
df["atleta_id"] = df["atleta_id"].replace(UNIFICAR)

## 4. Resumo e exportação

In [17]:
temporadas = df.groupby("atleta_id")["ano"].nunique()
print(f"Atletas distintos: {temporadas.size}")
for k in (3, 4, 5, 6):
    print(f"  com >= {k} temporadas no top {N}: {(temporadas >= k).sum()}")

saida = Path("data/ranking_50livre_M_2010_2025.csv")
df.to_csv(saida, index=False)
print(f"\nSalvo em {saida} ({len(df)} linhas)")

Atletas distintos: 1043
  com >= 3 temporadas no top 200: 416
  com >= 4 temporadas no top 200: 313
  com >= 5 temporadas no top 200: 231
  com >= 6 temporadas no top 200: 174

Salvo em data/ranking_50livre_M_2010_2025.csv (3200 linhas)
